# The story of the perturbation retrievals...



In [1]:
# conda env 'vis2_clone'
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.ExperimentSet import ExperimentSet
from src.Retrieval import Retrieval

In [3]:
def map_df(df, map_exp_name, ES):
    df = df.copy(deep=True)
    df['Retrieval Name'] = ''
    df['Order'] = 0
    for i in range(len(df)):
        new_name, order = map_exp_name[ES.retrievals[Path(df.reset_index().loc[i, 'retrieval'])].exp_name]
        df.loc[df.index == df.reset_index().loc[i, 'retrieval'], 'Retrieval Name'] = new_name
        df.loc[df.index == df.reset_index().loc[i, 'retrieval'], 'Order'] = order
    return df.reset_index().set_index('Order')

In [4]:
EXPERIMENT_SET = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08'
V08 = ExperimentSet(EXPERIMENT_SET)
V08.set_retrieval_list()
V08.load_retrievals(iteration=-1)

In [5]:
brf_ths = {
    'DF': 0.120,
    'CF': 0.076,
    'BF': 0.054,
    'AF': 0.044,
    'AN': 0.040,
    'AA': 0.045,
    'BA': 0.052,
    'CA': 0.068,
    'DA': 0.092
}
for rpath in V08.retrievals:
    r = V08.retrievals[rpath]
    r.add_threshold_mask(r.obs, 'MISR', 'threshold_mask', 'brf', brf_ths)

In [6]:
map_exp_name = {
    'default': ('Default', 1),
    '0p1': ('Bright Surface', 2),
    '1p33': ('Dark Surface', 3),
    '0p150': ('Low AOD', 4),
    '0p250': ('High AOD', 5),
    'mix22': ('Aerosol Mixture 22', 6),
    'mix54': ('Aerosol Mixture 54', 7),
    'unconstrained': ('Unconstrained', 8),
    'LCL_CTH': ('LCL + CTH', 9),
    'unconstrained_darksurf': ('Unc. Dark Surf.', 10),
    'periodic': ('Periodic', 11),
    'slow': ('Slow Cloud Motion', 12),
    'fast': ('Fast Cloud Motion', 13),
    'slow_unconstrained': ('Slow Cloud Motion Unc.', 14),
    'fast_unconstrained': ('Fast Cloud Motion Unc.', 15)
}

In [ ]:
def misr_S_1(y):
    return np.diag(1 / (0.04 * y)**2)
mask_conds = {
    # 'tau0.3mask': (1, 1),
    'threshold_mask': (1, 1),
    'ray_containment': (2, 2)
}
chi2 = V08.chi2(misr_S_1, variable='I', mask_conds=mask_conds)

stats_th = V08.cloud_stats(0.67, 'cloud', extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])
stats_all = V08.cloud_stats(0.67, 'cloud', extinction_th=[0.0, np.inf], tau_th=[0.0, np.inf])
stats_diff = stats_all - stats_th

chi2['Masked Cloud Volume'] = stats_diff['volume']
chi2['Unmasked Cloud Volume'] = stats_th['volume']

ref_key = Path('/data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/default')
stats_th_ref = V08.compare_to(ref_key, 0.67, 'cloud', extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])

stats_th['Relative Bias (%)'] = stats_th_ref['relative bias']
stats_th['Relative Difference (%)'] = stats_th_ref['relative difference']

stats_th['Default CF'] = float(stats_th.loc[stats_th.index==str(ref_key), 'cloud fraction'])
stats_th['Default tau bar'] = float(stats_th.loc[stats_th.index==str(ref_key), 'tau bar'])
stats_th['Default volume'] = float(stats_th.loc[stats_th.index==str(ref_key), 'volume'])
stats_th['Rel CF Diff'] = 100*(stats_th['cloud fraction'] - stats_th['Default CF']) / stats_th['Default CF']
stats_th['Rel tau bar Diff'] = 100*(stats_th['tau bar'] - stats_th['Default tau bar']) / stats_th['Default tau bar']
stats_th['Rel volume Diff'] = 100*(stats_th['volume'] - stats_th['Default volume']) / stats_th['Default volume']


In [16]:
map_df(chi2, map_exp_name, V08).loc[:, ['Retrieval Name', 'chi2', 'Masked Cloud Volume', 'Unmasked Cloud Volume']]

,Retrieval Name,chi2,Masked Cloud Volume,Unmasked Cloud Volume
Order,,,,
9,LCL + CTH,10.871575,5.082,13.211687500000004
8,Unconstrained,12.489859,10.050562,20.588906250000008
10,Unc. Dark Surf.,10.372845,14.758219,28.824468750000012
11,Periodic,20.778908,9.460688,12.534843750000006
4,Low AOD,8.661792,12.213438,16.289625000000004
5,High AOD,17.347160,3.198937,10.655562500000004
6,Aerosol Mixture 22,10.395386,5.65675,14.4821875
7,Aerosol Mixture 54,11.470972,5.687,10.776562500000004
13,Fast Cloud Motion,13.047756,5.709688,10.980750000000002


In [31]:
map_df(stats_th, map_exp_name, V08).loc[:, ['Retrieval Name', 'Relative Bias (%)', 'Relative Difference (%)', 'Rel CF Diff', 'Rel tau bar Diff', 'Rel volume Diff']]

,Retrieval Name,Relative Bias (%),Relative Difference (%),Rel CF Diff,Rel tau bar Diff,Rel volume Diff
Order,,,,,,
9,LCL + CTH,0.120333,0.337868,-0.141243,0.261946,0.749712
8,Unconstrained,21.124243,3.068575,10.169492,-0.040977,57.00692
10,Unc. Dark Surf.,49.033346,3.196247,35.310734,0.683455,119.809689
11,Periodic,-36.258206,2.266632,-7.20339,-31.310212,-4.411765
4,Low AOD,10.997113,0.566892,26.129944,-11.99781,24.221453
5,High AOD,-9.293176,0.508302,-17.655367,10.155114,-18.742791
6,Aerosol Mixture 22,-1.597794,0.778471,1.977401,-3.50587,10.438293
7,Aerosol Mixture 54,-6.222847,1.097436,-12.146893,6.743126,-17.820069
13,Fast Cloud Motion,-2.127001,3.449349,-0.564972,-1.570904,-16.262976


In [32]:
map_df(stats_th, map_exp_name, V08).loc[:, ['Retrieval Name', 'Relative Bias (%)', 'Relative Difference (%)', 'Default CF', 'cloud fraction', 'Rel CF Diff', 'Default tau bar', 'tau bar', 'Rel tau bar Diff', 'Default volume', 'volume', 'Rel volume Diff']]

,Retrieval Name,Relative Bias (%),Relative Difference (%),Default CF,cloud fraction,Rel CF Diff,Default tau bar,tau bar,Rel tau bar Diff,Default volume,volume,Rel volume Diff
Order,,,,,,,,,,,,
9,LCL + CTH,0.120333,0.337868,0.167574,0.16733727810650886,-0.141243,2.061031,2.066429954063104,0.261946,13.113375,13.211687500000004,0.749712
8,Unconstrained,21.124243,3.068575,0.167574,0.18461538461538463,10.169492,2.061031,2.060186616551134,-0.040977,13.113375,20.588906250000008,57.00692
10,Unc. Dark Surf.,49.033346,3.196247,0.167574,0.22674556213017752,35.310734,2.061031,2.075117373971214,0.683455,13.113375,28.824468750000012,119.809689
11,Periodic,-36.258206,2.266632,0.167574,0.15550295857988167,-7.20339,2.061031,1.4157179267412454,-31.310212,13.113375,12.534843750000006,-4.411765
4,Low AOD,10.997113,0.566892,0.167574,0.21136094674556213,26.129944,2.061031,1.8137525607094735,-11.99781,13.113375,16.289625000000004,24.221453
5,High AOD,-9.293176,0.508302,0.167574,0.13798816568047337,-17.655367,2.061031,2.2703312311641217,10.155114,13.113375,10.655562500000004,-18.742791
6,Aerosol Mixture 22,-1.597794,0.778471,0.167574,0.17088757396449705,1.977401,2.061031,1.9887740882413436,-3.50587,13.113375,14.4821875,10.438293
7,Aerosol Mixture 54,-6.222847,1.097436,0.167574,0.1472189349112426,-12.146893,2.061031,2.2000090934104177,6.743126,13.113375,10.776562500000004,-17.820069
13,Fast Cloud Motion,-2.127001,3.449349,0.167574,0.16662721893491125,-0.564972,2.061031,2.028654334351273,-1.570904,13.113375,10.980750000000002,-16.262976
